# Proyecto 3 – Mejorando algoritmos con CUDA: reconocimiento ASL

**CC3069 Computación Paralela y Distribuida · UVG · Semestre II 2026**

Este notebook es el entregable de código. Contiene:

1. **El código original secuencial** del Laboratorio 3 (Data Science), sin modificar.
2. **La versión optimizada**, con la separación estricta del flujo de trabajo:
   - **FASE 1 – Preparación de datos → CPU multinúcleo** (archivos, JPEG, resize, augmentation).
   - **FASE 2 – Configuración de CUDA y entrenamiento → GPU** (kernel Numba + red neuronal en PyTorch/cuDNN).
3. Una demostración pequeña de ambas versiones y los **resultados del benchmark** (tiempos, speedup y uso de recursos).

```
[JPEG en disco] ─► FASE 1 · CPU multinúcleo ─► uint8 ─► H2D ─► FASE 2 · GPU (CUDA) ─► métricas
```

El código vive en módulos `.py` (para que los procesos de la FASE 1 funcionen en Windows con *spawn*).
Aquí se muestra su código fuente y se ejecuta.

In [ ]:
import inspect, os, sys
from pathlib import Path
from IPython.display import Markdown, Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

def mostrar(obj):
    """Muestra el código fuente de una función/clase del proyecto."""
    display(Markdown(f"```python\n{inspect.getsource(obj)}\n```"))

import torch
from proyecto3.common.hw_info import collect_hw_info
info = collect_hw_info()
print({k: info.get(k) for k in ("cpu", "nucleos_fisicos", "nucleos_logicos", "gpu", "cuda_version_torch", "torch", "numba")})

## 1. Datos: la misma partición del laboratorio

17 400 imágenes (600 por clase × 29 clases), con la partición 70/15/15 y semilla 42 guardada en `data/manifests/all.csv`.
Las rutas del manifiesto se remapean a la máquina local. Para la demostración se usa una muestra pequeña (`MUESTRA` por clase).

In [ ]:
from proyecto3.common.config import BenchmarkConfig
from proyecto3.common.paths import load_portable_manifest, split_manifests, stratified_sample

MUESTRA = 20          # imágenes por clase y partición para la demo (None = las 17 400)
config = BenchmarkConfig(epochs=2, sample_per_class=MUESTRA)
datos = stratified_sample(load_portable_manifest(config.manifest, config.data_root), MUESTRA, config.seed)
manifests, class_names = split_manifests(datos)
{split: len(frame) for split, frame in manifests.items()}

## 2. Código ORIGINAL secuencial (Laboratorio 3, `src/neural.py`)

### 2.1 Preparación original: un solo proceso, una imagen a la vez

In [ ]:
from src import neural
mostrar(neural.build_image_cache)

### 2.2 Normalización original por muestra en el CPU (`ASLImageDataset.__getitem__`)

In [ ]:
mostrar(neural.ASLImageDataset.__getitem__)

### 2.3 Entrenamiento original (`train_experiment`): `num_workers=0`, todo en un proceso

In [ ]:
mostrar(neural.train_experiment)

### 2.4 Baseline instrumentado (`proyecto3/sequential/pipeline.py`)

Reproduce `train_experiment` llamando a las mismas funciones del lab; solo se agregan timers por fase.
Se fuerza el CPU porque es el baseline (ver `docs/decisiones_alcance.md`).

In [ ]:
from proyecto3.sequential import pipeline as secuencial
mostrar(secuencial.run_sequential)

## 3. Versión OPTIMIZADA

### 3.1 FASE 1 – Preparación de datos en CPU multinúcleo (`proyecto3/parallel/cpu_prep.py`)

`ProcessPoolExecutor` con un proceso por núcleo lógico. Cada proceso decodifica bloques de 128 JPEG y escribe
directo en un memmap compartido. El resultado es byte a byte igual al caché original.

In [ ]:
from proyecto3.parallel import cpu_prep
mostrar(cpu_prep._process_chunk)
mostrar(cpu_prep.prepare_splits_parallel)

### 3.2 FASE 2 – Configuración de CUDA (`proyecto3/parallel/gpu_train.py`)

In [ ]:
from proyecto3.parallel import gpu_train
mostrar(gpu_train.configure_cuda)
mostrar(gpu_train.to_device_uint8)

### 3.3 FASE 2 – Kernel CUDA propio con Numba: hilos, bloques y grid (`proyecto3/parallel/gpu_kernels.py`)

- **Hilo** = 1 píxel (x, y) de 1 imagen (3 canales).
- **Bloque** = 16 × 16 = 256 hilos (8 warps).
- **Grid** = (⌈W/16⌉, ⌈H/16⌉, N) = (4, 4, 64) para un lote: 1 024 bloques × 256 = 262 144 hilos = 64·64·64 píxeles.

Es la **capa de entrada de la red**: convierte el `uint8` HWC transferido (1 B/valor) en el `float32` CHW / 255 que espera la primera convolución.

In [ ]:
from proyecto3.parallel import gpu_kernels
mostrar(gpu_kernels.normalize_hwc_to_chw_kernel.py_func if hasattr(gpu_kernels.normalize_hwc_to_chw_kernel, "py_func") else gpu_kernels.normalize_hwc_to_chw_kernel)
mostrar(gpu_kernels.blocks_per_grid)
mostrar(gpu_kernels.normalize_torch_batch)
print("Lote de 64 imágenes 64×64 → bloques por grid:", gpu_kernels.blocks_per_grid(64, 64, 64),
      "· hilos por bloque:", gpu_kernels.THREADS_PER_BLOCK)

### 3.4 FASE 2 – Época de entrenamiento en la GPU (dataset residente) y en streaming (augmentation)

In [ ]:
mostrar(gpu_train.train_epoch_resident)
mostrar(gpu_train.train_epoch_streaming)

### 3.5 Orquestación: FASE 1 (CPU) → transferencia → FASE 2 (GPU)

In [ ]:
from proyecto3.parallel import pipeline as paralela
mostrar(paralela.run_parallel)

## 4. Demostración: ambas versiones con los mismos datos

Muestra pequeña y 2 épocas, solo para mostrar el flujo. **Los tiempos válidos son los del benchmark completo (sección 5).**

In [ ]:
import shutil
from entrenar_modelos import NEURAL_EXPERIMENTS
from proyecto3.common.timing import PhaseTimer, PHASES

experimento = {e.name: e for e in NEURAL_EXPERIMENTS}["cnn_reg_drop035"]
filas = []
for nombre, runner in (("secuencial", secuencial.run_sequential), ("paralela", paralela.run_parallel)):
    timer = PhaseTimer()
    cache = config.work_dir / f"notebook_{nombre}"
    shutil.rmtree(cache, ignore_errors=True)
    metricas = runner(experimento, manifests, class_names, config, timer, cache)
    shutil.rmtree(cache, ignore_errors=True)
    filas.append({"version": nombre, **{f: round(s, 2) for f, s in timer.as_dict().items() if f in PHASES},
                  "test_macro_f1": round(metricas["test_macro_f1"], 4), "device": metricas["device"]})

import pandas as pd
pd.DataFrame(filas)

## 5. Resultados del benchmark completo (laptop: Intel Core i9 + NVIDIA RTX)

Generados por `python benchmark.py --fresh` (17 400 imágenes, 6 épocas, N corridas + calentamiento por versión).

In [ ]:
from pathlib import Path
tabla = Path("results/tabla_speedup.md")
display(Markdown(tabla.read_text(encoding="utf-8")) if tabla.exists() else Markdown("*Aún no hay resultados.*"))

In [ ]:
for nombre in ("01_tiempos_por_fase", "02_speedup_por_fase", "03_cpu_nucleos_secuencial",
               "03_cpu_nucleos_paralela", "04_gpu_uso", "05_kernel"):
    ruta = Path("results/figuras") / f"{nombre}.png"
    if ruta.exists():
        display(Markdown(f"**{nombre}**")); display(Image(filename=str(ruta)))

In [ ]:
import json
hw = Path("results/hw_info.json")
json.loads(hw.read_text(encoding="utf-8")) if hw.exists() else "sin hw_info.json"

## 6. Discusión técnica

Ver el análisis completo en `docs/06_analisis_conclusiones.md`. Resumen:

- **¿Por qué los archivos en el CPU y la matemática en la GPU?** La lectura, la decodificación JPEG y la augmentation son E/S y código con mucha ramificación, independientes por archivo: ideales para núcleos de CPU (y la GPU no puede leer del disco). La red neuronal son ~10 TFLOP de productos de matrices regulares: ideales para miles de núcleos CUDA.
- **¿Cómo redujo CUDA el tiempo?** cuDNN y cuBLAS reparten convoluciones y GEMM en miles de hilos; los datos residen en la GPU (una sola copia `uint8`); el kernel propio normaliza con un hilo por píxel; la pérdida se acumula en la GPU (sin sincronizar en cada lote).
- **Factores que lo justifican o limitan:** alta intensidad aritmética y entrenamiento repetido (justifican); costos fijos (contexto, JIT, H2D), problema pequeño (64×64, batch 64), PCIe y la parte de E/S no paralelizable en la GPU (limitan).